In [28]:
import json
from pathlib import Path
from tqdm import tqdm
import random
import numpy as np
from collections import defaultdict

In [29]:
def read_jsonl(path):
    data = []
    with path.open("r") as f:
        for line in f:
            obj = json.loads(line)
            data.append(obj)

    return data

In [30]:
### Analyse is LLM certified retrieval docs are true hits

In [31]:
topic_format_edu = read_jsonl(Path("../data/weborganizer/topic_format_edu.jsonl"))

validated_faq_harm = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_harmonized_faq.jsonl"))
validated_faq_raw = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_raw_faq.jsonl"))

validated_legal_harm = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_harmonized_legal.jsonl"))
validated_legal_raw = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_raw_legal.jsonl"))

In [32]:
validated_sarcasm_harm = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_harmonized_sarcasm.jsonl"))
validated_sarcasm_raw = read_jsonl(Path("../results/LLM_as_judge/QueryDescriptorMatch_raw_sarcasm.jsonl"))

In [33]:
def check_validation(results):
    for res in results:
        response = res.get("response")
        if response is None:
            print(res)
        _, suffix = response.lower().split("answer:", 1)
        answer = suffix.strip(" *:!?.-\n\r\t")

        if answer == "yes":
            res["valid"] = True
        else:
            res["valid"] = False

check_validation(validated_faq_harm)
check_validation(validated_faq_raw)

check_validation(validated_legal_harm)
check_validation(validated_legal_raw)

check_validation(validated_sarcasm_harm)
check_validation(validated_sarcasm_raw)

In [34]:
def match_to_docs(results, desc_type="harm"):
    desc2doc = defaultdict(list)
    for doc in topic_format_edu:
        if desc_type == "harm":
            doc_descs = doc["harmonized_descriptors"]
        else:
            best_idx = np.argmax(doc["similarity"])
            doc_descs = doc["descriptors"][best_idx]
        for desc in doc_descs:
            desc = desc.strip()
            desc2doc[desc].append({"text": doc["document"], "doc_id": doc["doc_id"]})
            
    for res in tqdm(results):
        descriptor = res["example"]["descriptor"]
        res["documents"] = desc2doc.get(descriptor)
        if res["documents"] is None:
            print(descriptor)
    
    return results

def keep_only_valid(results):
    validated = []
    for res in results:
        if res["valid"]:
            validated.append(res)

    return validated

def get_target_ids(target):
    target_ids = []
    for doc in topic_format_edu:
        doc_id = doc["doc_id"]
        if doc["format"] == target:
            target_ids.append(doc_id)

    return target_ids

def extract_ids(results):
    doc_ids = []
    for res in results:
        docs = res["documents"]
        for doc in docs:
            doc_ids.append(doc["doc_id"])

    return doc_ids

def sample_false(results, target="FAQ", desc_type="harm", sample_size=20):
    # Convert to sets for easier operations
    all_relevant_ids = get_target_ids(target)
    results = match_to_docs(results, desc_type)
    validated_results = keep_only_valid(results)
    valid_ids = extract_ids(validated_results)
    retrieved = set(valid_ids)
    relevant = set(all_relevant_ids)

    print("Num documents retrieved: ", len(retrieved))

    # Find false positives (documents that are retrieved but not relevant)
    false_positives = retrieved - relevant

    # Find false negatives (documents that are relevant but not retrieved)
    false_negatives = relevant - retrieved

    # Calculate true positives (documents that are both retrieved and relevant)
    true_positives = len(retrieved & relevant)
    print("Num true positives:", true_positives)

    # Calculate precision and recall
    precision = true_positives / len(retrieved) if retrieved else 0
    recall = true_positives / len(relevant) if relevant else 0

    # Calculate F1-score
    if (precision + recall) == 0:
        return 0
    f1 = 2 * (precision * recall) / (precision + recall)
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1: {f1}")

    # Convert back to list and shuffle
    false_positives = list(false_positives)
    false_negatives = list(false_negatives)
    random.seed(42)
    random.shuffle(false_positives)
    random.shuffle(false_negatives)
    
    print("Num false positives:", len(false_positives))
    print("Num false negatives:", len(false_negatives))

    # Set sample_size=0 to return all false negatives and positives.
    if sample_size > 0:
        pos_sample_ids = false_positives[0:sample_size]
        neg_sample_ids = false_negatives[0:sample_size]
    else:
        pos_sample_ids = false_positives
        neg_sample_ids = false_negatives

    false_positive_docs = []
    false_negative_docs = []
    for doc in topic_format_edu:
        if doc["doc_id"] in pos_sample_ids:
            best_idx = np.argmax(doc["similarity"])
            descriptors = doc["descriptors"][best_idx]
            false_positive_docs.append(
                {"document": doc["document"],
                "doc_id": doc["doc_id"],
                "descriptors": descriptors,
                "harmonized_descriptors": doc["harmonized_descriptors"],
                "weborganizer_format": doc["format"],
                "format_prob": doc["format_prob"]}
            )
            
        elif doc["doc_id"] in neg_sample_ids:
            best_idx = np.argmax(doc["similarity"])
            descriptors = doc["descriptors"][best_idx]
            false_negative_docs.append(
                {"document": doc["document"],
                "doc_id": doc["doc_id"],
                "descriptors": descriptors,
                "harmonized_descriptors": doc["harmonized_descriptors"],
                "weborganizer_format": doc["format"],
                "format_prob": doc["format_prob"]}
            )

    return false_positive_docs, false_negative_docs

In [35]:
def retireved_ids(results, target="FAQ", desc_type="harm", sample_size=20):
    # Convert to sets for easier operations
    all_relevant_ids = get_target_ids(target)
    results = match_to_docs(results, desc_type)
    validated_results = keep_only_valid(results)
    valid_ids = extract_ids(validated_results)
    retrieved = set(valid_ids)

    return retrieved

retrieved_faq_harm = retireved_ids(validated_faq_harm, target="FAQ",desc_type="harm")
retrieved_faq_raw = retireved_ids(validated_faq_raw, target="FAQ",desc_type="raw")
retrieved_legal_harm = retireved_ids(validated_legal_harm, target="Legal Notices",desc_type="harm")
retrieved_legal_raw = retireved_ids(validated_legal_raw, target="Legal Notices",desc_type="raw")

100%|██████████| 7019/7019 [00:00<00:00, 1133041.60it/s]


In [36]:
print(len(retrieved_faq_harm & retrieved_faq_raw))
print(len(retrieved_legal_harm & retrieved_legal_raw))

172
565


In [37]:
# Only for sarcasm:

def process_sarcasm_docs(results, desc_type="harm"):
    results = match_to_docs(results, desc_type)
    validated_results = keep_only_valid(results)
    valid_ids = extract_ids(validated_results)
    print("Valid docs: ", len(set(valid_ids)))
    sarcasm_docs = []
    for doc in topic_format_edu:
        if doc["doc_id"] in valid_ids:
            best_idx = np.argmax(doc["similarity"])
            descriptors = doc["descriptors"][best_idx]
            sarcasm_docs.append(
                {"document": doc["document"],
                "doc_id": doc["doc_id"],
                "descriptors": descriptors,
                "harmonized_descriptors": doc["harmonized_descriptors"],
                "weborganizer_format": doc["format"],
                "format_prob": doc["format_prob"]}
            )
        
        
    return sarcasm_docs

sarcasm_harm_processed = process_sarcasm_docs(validated_sarcasm_harm)
sarcasm_raw_processed = process_sarcasm_docs(validated_sarcasm_raw, desc_type="raw")

def save_sample(sample, desc_type="harmonized", target="sarcasm"):
    with open(f"../data/query_samples/validated_all_{target}_{desc_type}.jsonl", "w") as f:
        for line in sample:
            f.write(json.dumps(line, ensure_ascii=False))
            f.write("\n")

#save_sample(sarcasm_harm_processed)
#save_sample(sarcasm_raw_processed, desc_type="raw")

100%|██████████| 1032/1032 [00:00<00:00, 970520.57it/s]


Valid docs:  5459


100%|██████████| 18045/18045 [00:00<00:00, 1076158.33it/s]


Valid docs:  3831


In [38]:
legal_harm_false_pos, legal_harm_false_neg = sample_false(
    validated_legal_harm,
    target="Legal Notices",
    desc_type="harm",
    sample_size=0,
)

100%|██████████| 1282/1282 [00:00<00:00, 342315.87it/s]


Num documents retrieved:  994
Num true positives: 301
Precision: 0.3028169014084507
Recall: 0.4785373608903021
F1: 0.3709180529882932
Num false positives: 693
Num false negatives: 328


In [39]:
legal_raw_false_pos, legal_raw_false_neg = sample_false(
    validated_legal_raw,
    target="Legal Notices",
    desc_type="raw",
    sample_size=0,
)

100%|██████████| 7019/7019 [00:00<00:00, 716803.09it/s]


Num documents retrieved:  907
Num true positives: 280
Precision: 0.308710033076075
Recall: 0.4451510333863275
F1: 0.36458333333333337
Num false positives: 627
Num false negatives: 349


In [40]:
faq_harm_false_pos, faq_harm_false_neg = sample_false(
    validated_faq_harm,
    target="FAQ",
    desc_type="harm",
    sample_size=0,
)

100%|██████████| 76/76 [00:00<00:00, 254809.84it/s]


Num documents retrieved:  255
Num true positives: 97
Precision: 0.3803921568627451
Recall: 0.21945701357466063
F1: 0.2783357245337159
Num false positives: 158
Num false negatives: 345


In [41]:
faq_raw_false_pos, faq_raw_false_neg = sample_false(
    validated_faq_raw,
    target="FAQ",
    desc_type="raw",
    sample_size=0,
)

100%|██████████| 644/644 [00:00<00:00, 637510.45it/s]


Num documents retrieved:  380
Num true positives: 122
Precision: 0.32105263157894737
Recall: 0.27601809954751133
F1: 0.29683698296836986
Num false positives: 258
Num false negatives: 320


In [24]:
def save_sample(sample, desc_type="harmonized", error_type="positive", target="FAQ"):
    with open(f"../data/query_samples/false_{error_type}_all_{target}_{desc_type}.jsonl", "w") as f:
        for line in sample:
            f.write(json.dumps(line, ensure_ascii=False))
            f.write("\n")

# FAQ
save_sample(faq_harm_false_pos, desc_type="harmonized")
save_sample(faq_harm_false_neg, desc_type="harmonized", error_type="negative")

save_sample(faq_raw_false_pos, desc_type="raw")
save_sample(faq_raw_false_neg, desc_type="raw", error_type="negative")

# Legal
save_sample(legal_harm_false_pos, desc_type="harmonized", target="legal")
save_sample(legal_harm_false_neg, desc_type="harmonized", error_type="negative", target="legal")

save_sample(legal_raw_false_pos, desc_type="raw", target="legal")
save_sample(legal_raw_false_neg, desc_type="raw", error_type="negative", target="legal")

In [83]:
for doc in false_neg:
    print("Raw descriptors")
    print(doc["descriptors"])
    print("================")
    print("Harmonized descriptors")
    print(doc["harmonized_descriptors"])
    print(doc["doc_id"])
    print(doc["document"])
    print()

Raw descriptors
['Non-profit organization; A non-profit organization that aims to provide decent housing to those in need, characterized by its charitable and humanitarian goals.', 'Community-based; The organization operates at a local level, with community involvement and participation in the construction and funding of homes.', 'Christian ecumenical; The organization is affiliated with Christian values and principles, promoting unity and cooperation among different Christian denominations.', 'Housing provision; The primary goal of the organization is to provide decent and affordable housing to those in need, with a focus on low-income families.', 'Volunteer labor; The organization relies heavily on volunteer labor to construct homes, with partner families also contributing hundreds of hours of sweat equity.', 'Zero-interest loans; Homes are sold to partner families at no profit and financed with zero-interest loans, making homeownership more accessible.', 'Non-discriminatory; The org

In [67]:
for doc in false_neg:
    print("Raw descriptors")
    print(doc["descriptors"])
    print("================")
    print("Harmonized descriptors")
    print(doc["harmonized_descriptors"])
    print(doc["doc_id"])
    print(doc["document"])
    print()

Raw descriptors
['Online Casino; Slots.lv is an online casino that offers various games and promotions to its players, making it a suitable platform for entertainment and gambling.', 'Casual Player Friendly; The casino is suitable for casual players due to its low deposit requirements and free play options, allowing recreational players to enjoy games without significant financial risk.', 'Table Game Variety; Slots.lv offers a range of table games, including blackjack, roulette, and poker, although the variety of variants is limited compared to other online casinos.', 'High Roller Restrictions; High rollers face restrictions on deposit amounts, requiring VIP status or the use of Bitcoin for larger wagers.', 'Bitcoin Support; The casino accepts Bitcoin as a payment method, which is a significant advantage for players who prefer this cryptocurrency.', 'Extensive Rewards Program; Slots.lv has a comprehensive rewards program with multiple tiers, offering various benefits and incentives to 